# Arrays and Lists in Python




---
## 1. What is an Array?

An **array** is a collection of elements stored in **contiguous (continuous) memory locations**.

### Core Properties:
- **Fixed size** — you must declare its size upfront
- **Homogeneous** — all elements must be of the **same data type** (int, float, etc.)
- **Efficient access** — supports O(1) random access via indexing
- **Cache-friendly** — contiguous memory means the CPU can prefetch data efficiently

```
Memory Layout (e.g., int array of size 5):

Index:    0       1       2       3       4
       ┌───────┬───────┬───────┬───────┬───────┐
       │  10   │  20   │  30   │  40   │  50   │
       └───────┴───────┴───────┴───────┴───────┘
Addr:  1000    1004    1008    1012    1016
        ↑
   Each int = 4 bytes, addresses are sequential
```

> 💡 **Why contiguous memory?**  
> If the base address is `1000` and each int = 4 bytes,  
> then `arr[i]` is at address `1000 + i * 4` — **O(1) lookup!**


---
## 2. Arrays in Python — The `array` Module

Python doesn't have a native array type, but provides the `array` module for typed arrays.  
You can also use **NumPy** for high-performance numeric arrays.

### Type Codes Reference:

| Type Code | C Type | Python Type | Min Size (bytes) |
|-----------|--------|-------------|------------------|
| `'b'` | signed char | int | 1 |
| `'B'` | unsigned char | int | 1 |
| `'i'` | signed int | int | 2 |
| `'I'` | unsigned int | int | 2 |
| `'l'` | signed long | int | 4 |
| `'L'` | unsigned long | int | 4 |
| `'f'` | float | float | 4 |
| `'d'` | double | float | 8 |


In [1]:
import array

# Create an integer array using type code 'i'
arr = array.array('i', [10, 20, 30, 40, 50])

print("Array:", arr)
print("Type code:", arr.typecode)
print("Item size (bytes):", arr.itemsize)
print("Total buffer size (bytes):", arr.buffer_info()[1] * arr.itemsize)
print("Element at index 2:", arr[2])
print("Slicing [1:4]:", arr[1:4])

Array: array('i', [10, 20, 30, 40, 50])
Type code: i
Item size (bytes): 4
Total buffer size (bytes): 20
Element at index 2: 30
Slicing [1:4]: array('i', [20, 30, 40])


In [2]:
# Arrays are HOMOGENEOUS — this will raise a TypeError!
try:
    arr.append(3.14)  # float into int array
except TypeError as e:
    print(f"TypeError: {e}")

# You CAN append an integer
arr.append(60)
print("After appending 60:", arr)

TypeError: 'float' object cannot be interpreted as an integer
After appending 60: array('i', [10, 20, 30, 40, 50, 60])


---
## 3. Python Lists — The Dynamic Alternative

In Python, **lists are used instead of arrays** for most tasks because they offer much more flexibility.

### Key Features of Python Lists:
- **Heterogeneous** — can hold elements of different types
- **Dynamic** — can grow and shrink at runtime
- **Reference-based** — internally stores *references* (pointers) to objects, not the objects themselves
- **Built-in** — no import needed

```
Python List Internal Structure:

  list object
  ┌──────────┐
  │ size = 4 │
  │ capacity │
  │ *array ──┼──→  [ ref0 | ref1 | ref2 | ref3 ]
  └──────────┘         │       │       │       │
                        ↓       ↓       ↓       ↓
                       10    "hi"    3.14   True
                     (int)  (str)  (float) (bool)
```

> This is why lists are called **"referential arrays"** — each slot holds a reference (pointer) to an object, not the object's actual value.


In [3]:
# Python lists are HETEROGENEOUS
my_list = [10, "hello", 3.14, True, [1, 2, 3]]

print("List:", my_list)
print("Length:", len(my_list))

for i, item in enumerate(my_list):
    print(f"  Index {i}: {item!r:15}  type={type(item).__name__}")

List: [10, 'hello', 3.14, True, [1, 2, 3]]
Length: 5
  Index 0: 10               type=int
  Index 1: 'hello'          type=str
  Index 2: 3.14             type=float
  Index 3: True             type=bool
  Index 4: [1, 2, 3]        type=list


In [4]:
# Common list operations
fruits = ['apple', 'banana', 'cherry']

# Append — adds to the end: O(1) amortized
fruits.append('date')
print("After append:", fruits)

# Insert — insert at index: O(n) because elements shift
fruits.insert(1, 'avocado')
print("After insert at 1:", fruits)

# Pop — removes last element: O(1)
removed = fruits.pop()
print(f"Popped: {removed!r}, List: {fruits}")

# Remove — removes by value (first occurrence): O(n)
fruits.remove('avocado')
print("After remove 'avocado':", fruits)

# Index — find index of value: O(n)
print("Index of 'cherry':", fruits.index('cherry'))

# Slicing
print("Slice [0:2]:", fruits[0:2])

After append: ['apple', 'banana', 'cherry', 'date']
After insert at 1: ['apple', 'avocado', 'banana', 'cherry', 'date']
Popped: 'date', List: ['apple', 'avocado', 'banana', 'cherry']
After remove 'avocado': ['apple', 'banana', 'cherry']
Index of 'cherry': 2
Slice [0:2]: ['apple', 'banana']


---
## 4. Array vs List — Key Differences

| Feature | Array (`array` module) | Python List |
|---------|------------------------|-------------|
| **Data type** | Homogeneous (same type) | Heterogeneous (any types) |
| **Size** | Fixed (static) | Dynamic (can grow/shrink) |
| **Memory** | More efficient (stores values directly) | Less efficient (stores references) |
| **Performance** | Faster for numeric operations | Slower for large numeric datasets |
| **Usage in Python** | Via `array` module or `numpy` | Built-in, no import needed |
| **Best for** | Large numeric data, low-level use | General-purpose programming |

> 💡 **Rule of thumb:**  
> Use a **list** for everyday programming.  
> Use **NumPy arrays** when working with large datasets or math-heavy operations.


---
## 5. Dynamic Behaviour of Lists

Python lists are **dynamic arrays** — they automatically resize when needed.  
This is what allows you to keep calling `.append()` without declaring a size upfront.

### How does it work?
Internally, Python over-allocates memory so it doesn't have to resize on every single append.  
When the list runs out of space, it allocates a **larger block** and copies all elements over.

```
Growth pattern (CPython implementation):

  Size 0  → capacity 0
  Size 1  → capacity 4   ← first allocation
  Size 5  → capacity 8
  Size 9  → capacity 16
  Size 17 → capacity 25
  ...
  
  Formula: new_capacity ≈ old_capacity + old_capacity >> 3 + (3 or 6)
```


In [ ]:
import sys

l1 = []
print(f"{'Elements':>10}  {'Size (bytes)':>15}  {'Change?':>10}")
print("-" * 40)

prev_size = sys.getsizeof(l1)
print(f"{'0 (empty)':>10}  {prev_size:>15}")

for i in range(20):
    l1.append(i)
    current_size = sys.getsizeof(l1)
    changed = "← RESIZED" if current_size != prev_size else ""
    print(f"{i+1:>10}  {current_size:>15}  {changed}")
    prev_size = current_size

### Observation:

Notice that the size doesn't increase on every append — it jumps at specific points (0 → 1, 4 → 5, 8 → 9, etc.).  
Python **pre-allocates extra space** to make future appends cheap.

> ⚡ **Amortized O(1) append:**  
> Even though resizing is O(n) (copies all elements), it happens rarely enough that  
> the *average* cost of an append over many operations is still O(1).


---
## 6. How Memory Resizing Works

When a dynamic array runs out of capacity, it:

1. Allocates a **new, larger block** of memory (typically 2× the current capacity)
2. **Copies all elements** from the old block to the new block
3. **Frees the old block**
4. Continues inserting

```
Step-by-step example with doubling strategy:

 Cap=2, Size=2          Cap=4, Size=3          Cap=8, Size=5
┌───┬───┐              ┌───┬───┬───┬───┐      ┌───┬───┬───┬───┬───┬───┬───┬───┐
│ A │ B │  →  resize   │ A │ B │ C │   │  →   │ A │ B │ C │ D │ E │   │   │   │
└───┴───┘   (copy all) └───┴───┴───┴───┘      └───┴───┴───┴───┴───┴───┴───┴───┘
```

This doubling strategy ensures the **total work done** for n appends is proportional to n, not n².


---
## 7. Building a Custom List Class

To truly understand how Python lists work under the hood, we can build our own using `ctypes` — Python's low-level C interface.

### Why `ctypes`?
`ctypes.py_object` lets us create a raw C-style array that stores Python object references — exactly like CPython's list implementation.

### Design:
- `size` → how many elements are currently stored
- `capacity` → how much space is allocated
- `array` → the underlying raw array (referential)


In [ ]:
import ctypes

class CustomList:
    """A dynamic array implementation mimicking Python's built-in list."""

    def __init__(self):
        self.size = 0
        self.capacity = 1
        self.array = self.__create_array(self.capacity)

    # ── Private Helpers ────────────────────────────────────────────────

    def __create_array(self, capacity):
        """Allocate a raw referential array of given capacity."""
        return (capacity * ctypes.py_object)()

    def __resize(self, new_capacity):
        """Double the capacity: allocate new array and copy all elements."""
        new_array = self.__create_array(new_capacity)
        for i in range(self.size):
            new_array[i] = self.array[i]
        self.array = new_array
        self.capacity = new_capacity
        print(f"  [Resized] capacity: {new_capacity // 2} → {new_capacity}")

    # ── Core Operations ────────────────────────────────────────────────

    def append(self, item):
        """Add item to end. O(1) amortized."""
        if self.size == self.capacity:
            self.__resize(2 * self.capacity)
        self.array[self.size] = item
        self.size += 1

    def pop(self):
        """Remove and return last item. O(1)."""
        if self.size == 0:
            raise IndexError("pop from empty list")
        popped = self.array[self.size - 1]
        self.array[self.size - 1] = None   # avoid memory leak
        self.size -= 1
        return popped

    def insert(self, index, item):
        """Insert item at given index. O(n) — elements must shift."""
        if index < 0 or index > self.size:
            raise IndexError("index out of range")
        if self.size == self.capacity:
            self.__resize(2 * self.capacity)
        # Shift elements right to make room
        for i in range(self.size, index, -1):
            self.array[i] = self.array[i - 1]
        self.array[index] = item
        self.size += 1

    def clear(self):
        """Reset the list to empty state."""
        self.size = 0
        self.capacity = 1
        self.array = self.__create_array(self.capacity)

    # ── Dunder Methods ─────────────────────────────────────────────────

    def __len__(self):
        return self.size

    def __getitem__(self, index):
        """Support indexing: list[i]. O(1)."""
        if index < 0:
            index += self.size            # support negative indexing
        if not (0 <= index < self.size):
            raise IndexError("index out of range")
        return self.array[index]

    def __str__(self):
        items = [str(self.array[i]) for i in range(self.size)]
        return '[' + ', '.join(items) + ']'

    def __repr__(self):
        return f"CustomList({self.__str__()}, size={self.size}, capacity={self.capacity})"


print("CustomList class defined ✓")

In [ ]:
# ── Test: append and resize ──────────────────────────────────────────
print("=== Testing Append & Auto-Resize ===")
cl = CustomList()

for val in [10, 20, 30, 40, 50]:
    cl.append(val)
    print(f"  append({val}) → {cl}  (size={cl.size}, capacity={cl.capacity})")

print()

In [ ]:
# ── Test: pop ─────────────────────────────────────────────────────────
print("=== Testing Pop ===")
print("List:", cl)
print("Popped:", cl.pop())
print("After pop:", cl)
print()

In [ ]:
# ── Test: insert ──────────────────────────────────────────────────────
print("=== Testing Insert ===")
print("Before:", cl)
cl.insert(1, 99)
print("After insert(1, 99):", cl)
print()

In [ ]:
# ── Test: indexing ────────────────────────────────────────────────────
print("=== Testing Indexing ===")
print("cl[0]  =", cl[0])
print("cl[-1] =", cl[-1])    # negative index

try:
    print(cl[100])           # out of range
except IndexError as e:
    print(f"IndexError: {e}")
print()

In [ ]:
# ── Test: clear ───────────────────────────────────────────────────────
print("=== Testing Clear ===")
print("Before clear:", cl)
cl.clear()
print("After clear:", cl)
print("Size:", len(cl))

### How `insert()` Works — Step by Step

Inserting at index 2 into `[100, 200, 300, 400]`:

```
  Initial:  [ 100 | 200 | 300 | 400 |     ]
  Index:       0     1     2     3

  Step 1 — shift right from the end down to index 2:
            [ 100 | 200 | 300 | 300 | 400 ]   ← copy index 3 → 4
            [ 100 | 200 | 300 | 300 | 400 ]   ← copy index 2 → 3

  Step 2 — place new value at index 2:
            [ 100 | 200 |  90 | 300 | 400 ]   ✓
```

> ⚠️ This is why `insert()` is **O(n)** — in the worst case (insert at index 0), every element must shift right by one.


---
## 8. Time Complexity Cheatsheet

| Operation | Python List | Notes |
|-----------|-------------|-------|
| `list[i]` (access) | **O(1)** | Direct memory calculation |
| `append(x)` | **O(1)** amortized | Occasional O(n) resize |
| `pop()` (last) | **O(1)** | No shifting needed |
| `pop(i)` (middle) | **O(n)** | Elements must shift left |
| `insert(i, x)` | **O(n)** | Elements must shift right |
| `remove(x)` | **O(n)** | Linear search + shift |
| `index(x)` | **O(n)** | Linear search |
| `x in list` | **O(n)** | Linear search |
| `len(list)` | **O(1)** | Size is stored as a variable |
| `list[a:b]` (slice) | **O(k)** | k = slice length |
| `list.sort()` | **O(n log n)** | Timsort algorithm |
| `list.reverse()` | **O(n)** | Swaps in-place |


In [ ]:
# Demonstrating O(1) vs O(n) operations with timing
import time

large_list = list(range(10_000_000))

# O(1) — access last element
start = time.perf_counter()
_ = large_list[-1]
t1 = time.perf_counter() - start

# O(n) — search for a value near the end
start = time.perf_counter()
_ = 9_999_999 in large_list
t2 = time.perf_counter() - start

print(f"O(1) access    : {t1 * 1e6:.3f} microseconds")
print(f"O(n) 'in' check: {t2 * 1e3:.3f} milliseconds")
print(f"→ O(n) is roughly {t2/t1:.0f}x slower")

---
## 9. Practice Problems

Try solving these to reinforce what you've learned:


In [ ]:
# Problem 1: Find the second largest element in a list
# Example: [3, 1, 4, 1, 5, 9, 2, 6] → 6

def second_largest(lst):
    # YOUR CODE HERE
    pass

# Test
print(second_largest([3, 1, 4, 1, 5, 9, 2, 6]))  # Expected: 6
print(second_largest([10, 10, 9]))                  # Expected: 9

In [ ]:
# Problem 2: Rotate a list by k positions to the right
# Example: [1, 2, 3, 4, 5], k=2 → [4, 5, 1, 2, 3]

def rotate_right(lst, k):
    # YOUR CODE HERE
    pass

# Test
print(rotate_right([1, 2, 3, 4, 5], 2))  # Expected: [4, 5, 1, 2, 3]
print(rotate_right([1, 2, 3], 1))         # Expected: [3, 1, 2]

In [ ]:
# Problem 3: Add a __contains__ method to CustomList
# Hint: loop through elements and return True if found

# Expected behaviour:
# cl = CustomList()
# cl.append(1); cl.append(2); cl.append(3)
# print(2 in cl)   # True
# print(99 in cl)  # False

# YOUR CODE HERE — extend the CustomList class above and test it

In [ ]:
# Problem 4 (Challenge): Add shrinking to CustomList.pop()
# When size drops below capacity // 4, shrink capacity to capacity // 2
# This is how CPython avoids wasting memory after many pops

# YOUR CODE HERE

---
## Summary

| Concept | Key Takeaway |
|---------|-------------|
| **Array** | Fixed-size, homogeneous, contiguous memory, O(1) access |
| **Python List** | Dynamic, heterogeneous, referential array under the hood |
| **Dynamic resizing** | Doubles capacity when full — amortized O(1) append |
| **CustomList** | A from-scratch implementation using `ctypes.py_object` |
| **`size` vs `capacity`** | `size` = elements stored, `capacity` = space allocated |
| **insert/pop(i)** | O(n) because elements must shift |
| **append/pop()** | O(1) amortized — no shifting needed |

---
*Notes by me | Reference: python-foundation-journey / 03_data-structures*
